# Pipeline Qwen + BERT: clasificación y generación especializada

### Notebook 3 de 4 · Sistema modular de dos etapas

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Introducción

### La idea

Los notebooks 1 y 2 entrenaron dos modelos que hacen cosas distintas y
complementarias. Este notebook los conecta:

```
                        ┌──────────────────────────┐
                        │        USUARIO           │
                        │ "¿Cuál es el 25% de 420  │
                        │   estudiantes?"          │
                        └────────────┬─────────────┘
                                     │  texto libre
                                     ▼
                    ┌────────────────────────────────┐
                    │   ETAPA 1 · CLASIFICACIÓN      │
                    │   BETO fine-tuned (110M)       │
                    │   encoder bidireccional        │
                    └────────────────┬───────────────┘
                                     │  categoría + confianza
                                     ▼
                    ┌────────────────────────────────┐
                    │   SELECCIÓN DE ESTRATEGIA      │
                    │   categoría -> instrucciones   │
                    │   didácticas especializadas    │
                    └────────────────┬───────────────┘
                                     │  prompt condicionado
                                     ▼
                    ┌────────────────────────────────┐
                    │   ETAPA 2 · GENERACIÓN         │
                    │   Qwen2.5-1.5B + LoRA          │
                    │   decoder autoregresivo        │
                    └────────────────┬───────────────┘
                                     │
                                     ▼
                        ┌──────────────────────────┐
                        │  Paso 1: ...             │
                        │  Paso 2: ...             │
                        │  Respuesta final: 105    │
                        └──────────────────────────┘
```

### Por qué un sistema modular

Es un patrón clásico de ingeniería de sistemas de IA, con ventajas concretas:

- **Cada componente hace una sola cosa y se puede medir por separado.** Si el
  tutor falla, se sabe si falló el enrutador o el generador. En un modelo único
  eso es opaco.
- **Se sustituyen piezas por separado.** Se puede cambiar el generador sin
  reentrenar el clasificador, o añadir categorías al clasificador sin tocar el
  generador.
- **Cada etapa usa la arquitectura adecuada a su tarea.** Encoder bidireccional
  para entender, decoder autoregresivo para escribir.
- **Permite reglas de negocio en medio.** Rechazar preguntas fuera de dominio,
  enrutar a una calculadora simbólica, pedir aclaración si la confianza es baja.

### La desventaja estructural: propagación de error

Un sistema en cascada multiplica las probabilidades de acierto. Si BETO acierta
el 85% de las veces y Qwen resuelve bien el 60%, el techo del sistema no es
60%: en los casos mal clasificados Qwen recibe instrucciones equivocadas.

**Este notebook está diseñado para medir exactamente ese efecto**, no para
asumirlo. Por eso comparamos tres configuraciones:

| Configuración | Qué mide |
|---|---|
| **A. Qwen solo** (sin categoría) | El baseline: ¿aporta algo el enrutamiento? |
| **B. Pipeline real** (categoría predicha por BETO) | El sistema tal como funcionaría en producción. |
| **C. Oráculo** (categoría real del corpus) | El techo del pipeline si el clasificador fuera perfecto. |

La diferencia **C − B** es el costo de los errores del clasificador.
La diferencia **C − A** es el beneficio máximo que puede aportar condicionar
por categoría. Si C ≈ A, toda la arquitectura modular es complejidad sin
retorno, y ese es un resultado perfectamente válido que hay que reportar.

## 2 · Objetivos

1. Ensamblar un sistema de dos etapas con los modelos entrenados en los
   notebooks 1 y 2, cargados desde disco.
2. Diseñar una **estrategia didáctica por categoría** que condicione la
   generación.
3. Comparar las tres configuraciones (A, B, C) sobre los mismos 33 ejemplos de
   validación y con las mismas métricas de los notebooks anteriores.
4. Cuantificar la **propagación de error**: exactitud del generador cuando la
   clasificación fue correcta frente a cuando fue incorrecta.
5. Medir la **latencia** de cada etapa, que es el argumento práctico a favor o
   en contra de un pipeline en producción.

> **Requisito previo.** Este notebook no entrena nada: consume
> `adaptadores/qwen-lora` (notebook 1) y `modelos/bert-clasificador`
> (notebook 2). Si trabajan en Colab, ejecuten los tres notebooks en la misma
> sesión o guarden esas carpetas en Google Drive.

## 0 · Preparación del entorno

Instalamos el ecosistema Hugging Face. Las versiones se fijan por rango mayor
para evitar que un cambio de API rompa el notebook meses después.

> **Antes de empezar:** activen la GPU en `Entorno de ejecución → Cambiar tipo
> de entorno de ejecución → T4 GPU`. Sin GPU el entrenamiento es inviable.

**Sobre la desinstalación de `torchao`.** Colab trae `torchao` preinstalado, y
`peft` comprueba su versión con una función que **lanza `ImportError` en lugar
de devolver `False`** cuando la encuentra más antigua de lo que espera. El
resultado es que `get_peft_model()` falla con un error que no tiene ninguna
relación aparente con LoRA.

Ningún notebook del proyecto usa cuantización de torchao, así que lo quitamos.
La alternativa —actualizarlo— también funcionaría, pero torchao está acoplado a
la versión de torch y actualizarlo puede arrastrar un torch distinto y romper
otras cosas en Colab. Desinstalarlo no afecta a nada de lo que hacemos aquí.

In [ ]:
%pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" \
    "accelerate>=0.33" "bitsandbytes>=0.43" "scikit-learn>=1.3" wandb

# Ver la nota de arriba: evita que get_peft_model() falle con un ImportError
# de torchao que nada tiene que ver con LoRA.
%pip uninstall -y -q torchao

print("Librerías instaladas. Si Colab pide reiniciar la sesión, reinícienla y sigan desde aquí.")

In [ ]:
import os, random, sys
import numpy as np
import torch
import transformers

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
transformers.set_seed(SEMILLA)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers  {transformers.__version__}")
print(f"torch         {torch.__version__}")
print(f"Dispositivo   {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU           {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: sin GPU el fine-tuning tardará horas. Activen el runtime T4.")

# Detección temprana del conflicto peft/torchao. Más vale que salte aquí, en la
# celda de entorno, que dentro de get_peft_model() veinte celdas más adelante
# con un mensaje que no menciona LoRA por ninguna parte.
try:
    from peft.import_utils import is_torchao_available
except Exception:
    pass                       # ruta interna de peft cambiada: no es un problema
else:
    try:
        is_torchao_available()
    except ImportError as e:
        print(f"\nAVISO peft/torchao: {e}")
        print("  Solución: ejecuten  %pip uninstall -y torchao  y reinicien la sesión.")

### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [ ]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

## 3 · Arquitectura del sistema

Las dos arquitecturas conviven, cada una en el papel para el que es buena:

| | Etapa 1 · BETO | Etapa 2 · Qwen2.5 |
|---|---|---|
| Tipo | Encoder-only | Decoder-only |
| Atención | Bidireccional | Causal |
| Parámetros | 110 M | 1.500 M |
| Entrada | Enunciado | Enunciado + estrategia de categoría |
| Salida | 1 de 11 clases | Texto libre |
| Inferencia | Una pasada | ~200 pasadas (token a token) |
| Entrenamiento | Fine-tuning completo | LoRA (0.4% de los pesos) |
| Coste relativo | ≈1 | ≈100 |

Ese último renglón es el que justifica la asimetría del diseño: el clasificador
es tan barato que añadirlo al pipeline no cambia la latencia percibida, y a
cambio puede evitar una generación desencaminada.

In [ ]:
MODELO_QWEN  = "Qwen/Qwen2.5-1.5B-Instruct"   # debe coincidir con el notebook 1
DIR_ADAPTADOR = "adaptadores/qwen-lora"        # producido por el notebook 1
DIR_BERT      = "modelos/bert-clasificador"    # producido por el notebook 2

MAX_TOKENS_GEN = 200
LONGITUD_MAX_CLF = 64

In [ ]:
from pathlib import Path

faltan = [d for d in (DIR_ADAPTADOR, DIR_BERT) if not Path(d).exists()]
if faltan:
    raise FileNotFoundError(
        f"Faltan artefactos de los notebooks previos: {faltan}\n"
        "Ejecuten primero S04_Lab_Fine_tuning_Qwen.ipynb (etapa 2) y "
        "S04_Lab_Fine_tuning_BERT.ipynb (etapa 1)."
    )
print("Artefactos encontrados:")
for d in (DIR_ADAPTADOR, DIR_BERT):
    print(f"  {d}")

## 4 · Carga del dataset

### 4.1 · Materialización del corpus

El corpus vive en `data/math_tutor_dataset.jsonl`, generado por
`scripts/dataset_fuente.py`. Para que el notebook funcione en Colab sin subir
archivos, abajo va una **copia comprimida** de ese mismo archivo.

La celda **no sobrescribe** el JSONL si ya existe: eso permite escalar el
dataset (reemplazar el archivo por uno mayor) sin tocar el notebook. El hash
SHA-256 que se imprime debe ser idéntico en los cinco notebooks; si difiere,
alguno está entrenando con datos distintos y la comparación no sería válida.

Hash esperado de la versión embebida: `cf4732834d64a196…`

In [ ]:
import base64, gzip, hashlib, json
from pathlib import Path

RUTA_DATOS = Path("data/math_tutor_dataset.jsonl")
SHA_ESPERADO = "cf4732834d64a196d49eda88ddac8694a529a8bfc6b843e8eb8d74b8c6d59ecc"

_BLOB = (
    "H4sIAAAAAAAC/9V9XY/jSJLYuwH/B2KABrrR3ZL4KaqAQWM8O4DvMLc3t+Pzi20sWBK7mmNJrKWkQlUfDPje7h8s4Ld5nId5"
    "2BsbBublgKkf4P+wv8QRkUkyM5lBJlWqKhWw21OimBmhjMiMz4z4py+K1RcX3he7w+b9zP/iHfx1vS72+GhfZcUWnyyzfX5V"
    "VkUmX8y0h3+kCWbwaF9cl/hKeZ1X2bIoafC2uMnX+PQy2xXLEh/BqKt8K5/l+CTfArAVzf+HfHfI1ze5t868XXF1KOC73JNT"
    "3v+yvfCCKPbeeuE8nRC62boQI7/LdqXnX3jfA4abcucdtvDFKt+981b5Mt9mO+/OW+Js+CeM2ubZdlUCnJ23LD5W2W7yX7c0"
    "R6DA8L70kiCEbwCv60O+22fex2KbrS/wMcK/hhE7AP9fvjgeLs5jQvziv8HDqgb6RwKKv1KCvcnWZaV8znd/XOUbXP6P2XqX"
    "/49//+/+SaVs8CIoG07muAjBJOJo+xUtH64yTLIsNxmucbHJ1l4Gk+3gk4c/o1JIqUyKSzthqDkxyekMCodZgLAEnJgUnDiR"
    "MDwNCa+r8nKdi9fGUvBvD9nWq/JlWVVAMFjZ2Pvvxfr+l02+r2CdrssK12qT3f8l22bA9cEktb2wz6pVPvF++7evD/c/bvfw"
    "hfpSO3++9fblPlt/sPPCN2vxtZcTifC3I1FwX61gnlUBy75dFtq+RpTfEl5IJYYTQhUfnSncYQqu0KHxLNEBqXCHC3NET72/"
    "t4f8phy9wf0gnuEpF6WzgcN7XYoV3d7/uslhTRQaKpPAukbzcGYjIz63n8+dqfE1y6QcteqZGxLVDwZoFL8MGs3EUTabJAyJ"
    "vtktq+KywKWEl/BoLPGPmXedVZkHc2brrBKrLA7MXKVeOz0utD8JYxv18Lmx846AijNY4HGErYE2hK0fDBA2eSmbDw8j0jJS"
    "x913XRW0RdrBX4KKolCzHpDDuZhXS3gXztg5gYBX51bazuP+fckAxUGD4DjSznXCzl3IOn9igWuj6j+CHL0sLtdFuc+XIDnz"
    "7f3PmeeDrrguLlFa3qG8hH0B4jJd1A9ppp0mYeU3e2CL3Ms+lVX2gdOw5IyVPhutOYm+7LABDLN3Xra7/9n700HoQRv9jCZt"
    "FjD60gvCyMYF8FgCMNWu4+HTQa5D5hhCB98wBjx24IxU44wbWsO93LhPyh5fZ5sC9jvSswIO8OPZDA7EfCeWa33Y5sghAcg2"
    "5THorKC/KtzRjB9WuOjN0q4Dye27Ag7VWIGQeiuxwK0M/7VxBH0h0OwoXc5wiQWsEDlOMMA2rEDPHZhhcS7M8BVYnh7+tlVR"
    "evS8Kreen4Jik60O6z2dFsDgM29b3P9FPR9AewU5sSvRQG0GDjDD3x5gnHpsX1WHa5K+zVwqE6SkXhF0UK/8hV1ng+fNaJ0H"
    "XMAR7buAWD3OhNYqdPDNMOX92WPK/QLEd7XJgZankP3RbJHAsiyC9Hi9u50DVjaeBdZjHZ+PULu7c3LUqiduiFQ/GCCSfyZS"
    "vDrs6bja5dtVXhU7UF+FMIZNKzQeD96Hp/BOMAl0y/oO39rlVwd0HDWvkdNBea3e0PUJuS63V8X+sJLHJtmphMeHAR5QhioH"
    "LMHVnGQToU0iGmjg2oxp3pR2g0auMQMOa0bzRrQDowQvZzf76WQBCzKfzOJhG41eJnMJ/3K00sSrAgRqUvFkYdXl6QvWUBsB"
    "WpzdFqCsAldDbnW3+skAocMzORFw+68y7wb+RfU9mntLUOd2OYpjUNN29z9eZrDd77wkxM+rclNsr0pNarcD6llWhwrdkzgA"
    "lkocOBvwxg1ueZoCZ+pR5SI0tQAbsKN9qwCHxy1O3HbnIZFE0GCwproJqLXZfRfRHZ1QaXuKHT9PyawJ/Dg+XoArk+DygpJs"
    "JyJ+MUKG26Zl6VbP3dKrfjJAsfgMdu03W5DDYJCXq/wq8z5ldyClctyFPwhuho32OUMleomKdwBwFzHu31kTE9J2b/M6zjSg"
    "bqvbp3WSZBI4ukrw5FzQuenHFg8N+t3JZZLRkAt8DXcaqsmB3VEDjxsc+c3sgI3quhnCgz3vDWTaY3/ImQOTuYU2K4Ro4R//"
    "6SJgKbi83ntRvGB2+R8QRVxGWxwRWO26WKFUKYEiZAmCnliCvgkcBgwLC7PPFNZogKGNHFuDIfBYp/wJEMAJTdCscR7rsTL5"
    "eYDYwQsh9iSCRQgnwXxYi8N3SZOCPxx1OHpTzI/G1cS3UhifswqcM1SiaRcea8xJoK0xJx8MEDZ0Fdsu1LWKAk5sWwj8LZ5k"
    "G6TnHpQuD080YPQEPBMZnolf5+tdcbBERUEMLKW2tqhfrtU2eHFbLj/lrSn38ZCLM7OFRMvIyIl/3Mr54eMuB1cq7kNUtzxa"
    "EtiZCockQC7cge/Dd14uyFm9EzJN4nWZ/VB66Ge3sc770PitOh85oIIDxiDB8ZMVk4a53ruwVvTkZ8ZxmiD5Nt+DhJ3HQzKi"
    "RxVUZiEHaRDbPbJBzBz/Vk3QMivvdQ1iw90auMjx+IWQyaeVgLCVi4UO7worGTRnVwOdXhUQKGNoEtuTlCY95rkzXLLOLRD5"
    "JKVJbGQpTVyom5yIuieNuZGXLgSNtgm5Cd0Gg1y2GBtEpFaQJwO5INflFubJd8xZ/W07ptaWdnhS4ql5wIMSD1AZ9VJzWALa"
    "ZBFZW3bNHR5bg2ujIFIGiwGLtes0gK1x56Sbz8+D6pCXtMVoIhhvguaYEgJIVSLMkMHOANrDah32xboAQ8RLU/m9wQb7lg0+"
    "DJzRsNo0sZwT/EHgqUUMcrAYS6/YFpBNpFIfkXqPkIEgsKtsxE8CiQNzdjvCJNerDo0lvwayJX8SOJA/fdQj/cSe15kQcUn4"
    "IPGrToOSMmHkbzJO/tqmZQVwYgrgxEkAL85hs361XVX3P+3qfIhIiXvfeVfZbg+6dZA2cV011r0CsxUiyetcbFA+CwKngfdK"
    "oa1u6BBY18O7+zKSyo8Ai9uFCXPTF7YwtytM8pJaofHb0x7ipufDNDcCnQ80u57EXTpB98JskjpoX/guaUHwh6PyRW+K+UVm"
    "Z8wkkvK6lzNU8rB24fWkksZm/qjDtvb9x9zWY4j7ffkRt/SmAJHrT5LUa0Kgu4P3CabJtmX9bRx49sjnqviYVzkm4qLkRrSF"
    "iQ5gMDQCK8ypZNpYkPXlpdAFapMVZ2/mgzwkMO4zYd7flWrWOaH+XuD4JVDOt4ZH8blnC44+EA8SBl0MOKYx0Gh4B5878E7w"
    "QmyyZIb+NvbeiIvgrqdAB1dgJSk8HiOyjQlZh5mYtfWXBS6EeXx32RjtOs/2VZ30gPIIhuZbkZUExj9yM0hor1wertEs0VTq"
    "5lVpW6GpwZpV2oI3I+t5Udc1LapYGuu4UXxIkbFJbchQqefqoW8POOkg0QCxAtuA1krsIHWge/SCNOr3MXmMmD35HeQqFkIP"
    "wjcx2xWYaJnjr99hfnAdWMP4GWa35Eug2fawuf+pKpZqsEPCQUejP7d6M/25TtiHwsbZdKis+1KAbh2W4vMAleNzEdkQFv0P"
    "5VUJ29f0VItrUn5k+sQp3Ama7e/zGwoRwZ9/OBSf5fsdb26/eFcAOkn1fGeiSUKTpCh9ReJUS2YGIr5+H77BXRthJhJ5Q+yO"
    "l16X+FGIiKxmOwq8P6bPIT7EXpvD2i1iCi8CF62B25emWBF8Fjzh5cHEu/+zhxmEVh74Hbi4yg145HD3tm+TmUx/0vUf+jNS"
    "VX167kf4YppYUx7hsU7lUaBIvzeAcFSVkBo6ys8DhAyehpAjImcOqhr4mGF5/AknGP6uQRgPY7KkdgXEzrxPh8uCkgvgxiYs"
    "fiLmwRN4ns4Uwn6D9wx2MAseQEIpwe0iA9mN/YXGHl0Jg/9YL7NMUp34RyKGkxyDEnvhReDV3niZpA6sEp6aVR4aYwW9MTvs"
    "y839LzfF2rspIGME7BxMwoG5yiXKX/Rcok20Rzc27NfOjVO87DJw2zSvQHaBSAroZdJIZ2BkbgE0p2V+XW5vcqkgoMu8gGMc"
    "blKICS6sM2GGC/A1faFw4u/qS6PwffurgDnknF/ij8Jjg+Iufmr3KgFHsKm2D8UVZxuPJSubuqi2kimdObCpc/bew3nVUTx9"
    "l68qeY8mw+ynxh25RPe6SMG0eCE/ZZfIeWIcKUKwWujYBE5O5TDezLnO93BLCm/0UDq5vCNzk8GvwxzLJWqmkGXWrIGm0xCO"
    "QC+0PBcJkwpIX9h8lccAF9dsO2A5NjFgNyxCzx2YJH4+/eXoy9G0Mk4Sr/9+dL3Cfk+KZ5/Y4q5Ia/P25Xh2UjwdCJa8NIJB"
    "qjKsRzhzIph8mfR3H45aTAdaFaJeSJ1ho+RZ08TiZTsFI7+XgsPgZLK1BoglqYTWkjTyXUg6fxJ9YoQTKltvsuX9T1txxxRM"
    "01hm796Jkxo/4NWTHPX19f2P18UyN6K78mFz7/aDC/XhnJR7CY92AZISw9QZ6ZsVqTfa/UnFfLBfnITk2XqSPp4YjYS4TKmB"
    "58OJOg5KWNGFUdJHkehP4b8KhKUSTSKnY2AHlzKkBSCOUzL6fH92lG3i+xMR/vPttzH6uKEHk1HGiIIDf1vDuKLhwBGL59Xx"
    "bMzwrYggUcIEHqObwy6HxESx5m20mfaUvFeq6nzXkAK9hZzL+sqp68FxjSdNSbPqO9hyz5awAFomGE/gpD99YdPqjgBNib8d"
    "oGwYw7frdLGbiuB04/a8dLqZNPPdvFLN66Rjib/JNST+DrQUEnHR2ae9F9glAz7v9U0NAhRpJSYoPm6RGHpC4HL8+/7Z6QmH"
    "VbGHt+qjbw4rum7VBPwgv/HBTivW8J12tUc8crnY0yeiBUyRQN7MWQto/FJ1VBLtUkEhhhfkFO46wjAC5L7UQPfwhgJf5RAH"
    "BnlmH+axlZSEYE041YBeyP90KGA1ciFTlgeIOpVKbZNVcVMIxT2JZAJDpJVTqmFQFpA94Ugn+NFQRTklHR6fcmRkGjkQ+Xm9"
    "j8xJcHlAR8ayxMwc6SwEKav5EGk34A3WIS8jngUQvaHLrq7HgSyHh86hIquswti8ppuKc1yocwvmou6ix194PBJCHdDB81rh"
    "oscROFiCg1gmeonnAtYmQeEaO4aq6tfRBJN/i1p19Ld62TNJqfCNPyeNPGUyENN41q8VDEKkzJUuLDZuFRseoTR2UvfiMzwQ"
    "tHv7OdxcuizXOyEVM/CAfhb1O+iAKK8qEQSq7WnjeKi/l3f3j1AQJGy5G1GvZpwIi9Y1HzBXRQIda3clYQAJnMgEz17y7eLQ"
    "3vMdvEICQsstbI3SbWfnlvBEAYG/yypMYxQ6IhSFBQd4tsnXlO0EohdSAPBCHGaYVGtpMlL+pxS2iQfFDa6MLKh2DlkhDUIH"
    "tMz0Lh8mICgoe66p8pcAJPzzQs6rgQHA9f7/kilnjXcqv6QTEXCAQ05HBQJ7Yuhg2oPDgQOC03HAY18ABi0KloK3DhstrMfd"
    "30xCIThrjDowTnt+WpLf+oRshDnQA8yBA2nCxybNSZMQoHJBgisxc/TtZZtLuZV9mdINOx2KHVP1Flk7WuTzwy8sKwRA80ea"
    "t1c8Qsm7sDp7F30n8/EoCF+vCpx19S50B+/CgfDRI5/K41IKQJvPr0CLLSvwyMENKEooyG+Xh2qHlBcLB4XJ4HeDH4DKfU+8"
    "r/GkRWtAnOnL7DoTMXB6GxI2tbeVYxvG5HTLbotR0QIv2dEQwGgLVwQrJA98KldsrkGzYTEiixSicmnhJAq9DJbjFsi6yrAo"
    "Zq5w0u/p1jUQD4TxYQcw8EeC83ZJ3E2C+7BTTL4qp0oNxEafMgz0gz5TXEKFjkj8BGvFPvEVc74Mo4sDT48oW/Gvxbat9efA"
    "vfF56BQuSQaivkCO2aiQnlqJ1zrXoNCZI5ILGqb8MCiCMB9x316iqEP+YqK1mVswEzJkJmIDTBS6xoKVTwMwm5QCExofm1ZB"
    "KhFqBy5InkOvOP6aIm28+CGaRTsLltK0V+ycjVItzBnZjWoUWHchz9w1YHTqncpWimrkDLqBUbYqEkLULa6NgJzKrUZ1UVOp"
    "OGtSxBiLtyBgCO18GjW4f5v6yc1mUmw4AXicHqKixHGBC9AxmocJc5wWkj6X+nnsLcaYNgtbQ4LX/bDKujwAlBBB3Ow9e7En"
    "V81SmZ38/+q8LO10cjlQa3FOG/p7YN4NaHyw7QLwgl1l67Uo1AgeXQjpZrAlFZNXk7TNu/Cfm0xsWTngYZu2hqr1LRKSEK1r"
    "314aIm4QOmrTNkBF0yINHCt1dZit2DUcOvvqYHKBU8j3TKTuIqKNMH+IzK3nwAVlWmOYt4oHZK4xI0sjs/dF6OBt85/D28b5"
    "ZT9iMXywBuT9/zCeKdXwtx6UDwTFGAwJVCohZxV3IhT6oAxy3JIbvSa+nOWa7uPsXDaquLzP7xwCr9ZxiWvFFU/OOVeOcz5j"
    "sjWOAE4lXSxgWR/PzJ6vMR8u2knsEbwsgeuLdUkng2Y4+icxfTCoU523NBguqZeXYNZrLh0tkyq/zZZAqBwvENr98H6nZNMo"
    "uJ3cLQMiH543Sjb5Tv529zvGj+HeYSS1cLCjgnyToUMhjZQ6Pj8crrA9AoQt9nmdz4CNusxKPvBN8woo6iC+tmw5H20r3pRr"
    "7FCibUZsIdc4jOroiJxe9eWS5zUUgRJ725SowarnNBiNAvl9deBsjEbHoI3POHhS/OhlHQhRmtCJsGDblRUb+DEfpY5MvlYs"
    "ziBzrJHNSG+mmWgiLdIiHmHWnL2XgtlJYTQwEW5RwbCZeUZzBRdixs/k1GUyNWCOjyCpqjq+9rGEYiEVZbrITiG6BY5Fmin3"
    "pRIRyzhkXbj1HBgkBzVWFE2j+bfDLluYt453EUxY/WJ1KLXA/fflmjJJqQTb2gZRbXcEk6TNG3e1OyDWfh2m94pX7GE8OZyV"
    "NDzWVM35MfHlg4IK0q4hQUDILSjcnCcwFwy6BKgrccQZfBs9XYDQp8YX6RyLRvSUhC5hElmooPhM3E8pytBnJF9+ypRpsBy3"
    "mkRa167AKtwBAaFr5aH9anu3FPRIwDieBclfZQ+Ny+uhA8WDp6X4SeOOGCD/s0dl0cGlPWFrGqipLxgpEYEpqMUO8EHIKxNh"
    "IHfWLQrfXG8PKIXH7uJNTR/vEZCVAvBdmOx2N7zA6ZAbmAjvrI+e435/HRLR0zdtbYDeDS9bJVLeDSSc/IQnaYEJXGIaFPmh"
    "QnbNoyeO+D+Tz1WcCl1l0w/YLe8EGkf3AmWVTF/PLpCfB2gfvdBjfiG0TCzXxUZsqHQI6lLW/dbM8CUdGbYT3qcacuSltffy"
    "MRv5jIKoH+06LPaGr25wDjXvIRLHL1WSR8j2r0nkxW+O39liAir+wO5rAQuF7sKaw7VIR+9qBaxlT2sAWWrrBSkWLnpb8pIP"
    "c1Hsy5+LfqTHa2/tPBBfCWKbGA+Dpulp5Ef21orRaO3NBKxJcQMk32QxMlorRg50n79g7e019nsHETiZkxQPhwlvbrZmAlzb"
    "Scxv9EjcPaVb3iHjTQwnMUv4IcCWrW6C5MMJpjsxnLic7+kLpryPq5eIo7Cnpp1Ff4awoDx5L+oJgAKGpTZobCH4CLM2k5no"
    "OiVLz0VWKRClg2o9h5YU9g9BiD0vdCkRuUiJxTNIidMeGRGWCVy8ER6Xt3SVeuyhIaagW/pabaHGoRMmtUOneSFURIiA21PY"
    "zvkUUTERlYOsOGjSRIXOOwJGFbFD1nCKHp8rV5Ch9Br3UPyGmjCMZwoaLI5qVogQGD+sE31qxmiVefDQvBfXuQKmB6gzayj4"
    "WGSLholuUGg48C1BjX6gDgziv2RNQyTqC2/Bax+33dzBtJAJKApZ5Dzk68VMNZxIXgBhmCatKwHZSyImvLI5ArqFRVS4bAl0"
    "XeNMHBRO/9HchQ/J95XF7y7vf90pyeJ4iMLKZj9Qhof3DdWelQ0YKbEcw3pQ2wV97k1Xd6yefolJEFpYo5lGySjEJFsJ1KW+"
    "Mvr3CRod7u+JNkFq6z8pXyY8LjAlXBwFoT0VLWyR6ym3zEDX+k3ycNlUNQN4m7Xm4Hl2usX8DKwkaxu1zW3U+kbfF1pfm0Vb"
    "DujOWx8AIfw6u6pyrGACKhxF4CDE89u/LTt1kTDZFdO0M1kWGcG6cFIzAdwBWFF8IJJ6hL2fKeY5ERoXtaucOojaG5ky9ZHc"
    "ETBZygqa711qr40UuKgw0ZPy08i6z1gFB7q5Yp7JLqfgH2ahhFhQAMyArTAOoli/qXDYYlfroO5EhYHZWecNNRcLr4dkV5CN"
    "NVBL4+saqtQlItkcJYy18krfElyse4mFDOSdgWDWvhN22ylTftXuQswlwhc0LoiZS9b0hS1tqxdJfJVHz2TCYcR4VcneEYae"
    "OzBlfB5q05F51QEqHK8joVkf56OJpBhZ2I0tAWKhKlGhXYkKe1J93E2uBhvD4FLx4NSpcCjfZ1Saz3W5d4u3w4tUBn33R/ha"
    "nowGY8RP6Kz97V+xwMFv/8qwwtfZenlYN02vG+TFSNxuWD6X/lzYPLTkLSWj2Err0KD1KHCaJa0CYhUcnaShC0mDF0fS4Ldf"
    "+sKo2hI3KyyGDds+dFPFGjQNgx5SWuBw21JCYGmoB0jDwIGG4Yuj4V//5X9hWfy3bE4ueC2r7P7nz6RxkO1DDScike+K6duo"
    "/PtBW+Usimz7E757K8wHuzTv5kCMB6vtUxUge/TqCRDBzIHG0UukcRCg1Q9/RAu38xcXf4nVJ+Vg1NHhRKQZjOCoEpVWHRvd"
    "PJfeE3gQoBEAV7wYXHrLmCw2Im38VKQ9rZdqAWIKkwHeiGsob6Xv1/1EXpCcS317OKTjY/Tr1INYdXS3+dHoC5GbUNw6nEvX"
    "kwjf4F61V1x2O9cbbI0oSS+emtrmhCFfqtko1OzAWMlznhlHZmb/9gvX2hFv2VXU0vZwWRpF6NVbMdRinq5RoAJNEtdo/hLV"
    "chguuN3hP/Kjm9PzwYhQ9jaPwon8n8QB85fHAXD0whZqipQ6qgYwBBZ7wWt3olQVtbCw99lwUARqIBbVTp2eb5Bh9MRwIKBz"
    "Ae1HFRDHXrsiCZEcYXzR0C/lHaiE/tbin4pEFq33EuftOxaooQBo4E65VRcvmNLhX//5f08cy9yRExE7RsqTMax9FQtY9UXj"
    "uFBVvaBpepBaC6Kn/kBVO2eYdPlZhcZqeXrp89QfprBT0Pr8DmM/WYB2An+EyXgdHgejD1So1KGWpaCZaWGtAVmLUvgLZyXe"
    "ClG30BRY7FGt18LyFw7k9V8eeYO//vP/cd230Ne62C6bLSSaQhn/sLdjd45OleOgdu/G7k7sYjECyS/rdMZdQTGE2E0SC82n"
    "GSfLPNWE5eyj2oFpS0Ki2vOikIg11cQsfTCIDr4+iIi+61UUWM7QM02GSiUQZ4Qvb9ujbtMUB3S31y16WCcHLJKO8m5yupuV"
    "raldzOxsJrqeiO5Au0dwqj2o3IXSDjdXzzNsdi+sTnAvkuFJRglXHEq1UMWLPVYqVsLAG53U702r9mb4Vi2ekx7LeCRcUfFN"
    "d6uynhDTFeKgXvvP7mQ7+uyOZG21MeYwjoGl164MKbspEDP2VPBzsIZbGMZe1WbnK/aNqtQHdSPdwo+iwCQGpy3US54wSjWl"
    "DJIpe++LMiS25QbD5yV6ivA6tawzpV6ylnkBO7yZLTJUMCOMhqgmEgLDk/MdloUot0vzCEGVaGoXvlPzgtgpUCNLyhEpVhxP"
    "DXk8jR24JHh8LjltPvAU85H8KRcKU7u84ruZV0JDespSwo9g30xTq2sEnouJ0Rc9tbq94DHfUpYFprtETDBsKYqp7v+Snweo"
    "Gb4wasZTynmfhg7UhLc8LPB6KzJ9puSEmFoNZDEtfIlBqam1Hww85mnJgdJ0ZBMIWz5qqneBkZ8HKBm9MErOpz7ddmWPcHV9"
    "8bjHZjE/LaUMnFIp1Klvv7Qr58bv8Xyc2sUwPudJ2gNS36AWYPyJa8jl+sEAbeMXJJnDqWhsFbl6PhqxVvfCwCnx7lTYNMTR"
    "7nQpJZvqim2KOL3Aq1My9mfdx8lQiw4ndPTI4zAirCtb92Q78ELywvZ5NF2Qvsoe2WIZxXLrNcwN64ZMmoNX4B7dZXj/a7oQ"
    "lzjCafBG4ZC/vwRDg6LHfjD103dCnWrKQAnNCBGypaxMQ1txn4ejh7M6I8amtkz1qi7y8wDLzE/JMg/Np78uPn9GUSmam1Bz"
    "rxTXTgBvS8FCcv1BKU8JGfigEEO5qfZdTKT/E/R916ver2sQdDlD9mvFsUX1gTXxxAhZhSlDKqRTzByg7GwCvJU6XlfYpKSg"
    "xaSghXY9EB6rqHWsP1fwuuAxAfNSpwNdEUAuWmL66Aw0rhMFpcB79Y0MyMi+//GqEBd2vgKe+ZR56/z+Z9hOYNJQ4w4aoBUp"
    "bIY0L39wcdeZrFYXQ8UiNFORFh1Es25pYdIdkKRR2nYY59w+DXJ9Tj1HTJR6wwwOPY4gDRF3jxDxzOL55NSR8fRp0GtafKOJ"
    "d9yt97/S4geYVIybNhGT4G6ktJM+iyNUjIHYbnHEpsXxAAz0IhQGbN6kTAyT0kFBMeKwPbGcR9BYj4zTTUXEeuqmsyp+GOob"
    "orhysI4FTfVm+lpcU3lDRr1WPk4vSCl6fOVYklI2MbugAbg7p9aoHjzu02GPQ48YZAxi7LEx1eN/8vMA0/gv7bRAsYs13B15"
    "RjQDVpRDGC+UQ7Ab31A+s6ZeDJAiuqABtH8D+9nRyyNu6DjwhI4If5AExkHiwhPByQ+SUzQnqeueJjO1EOc78g/ANb7lvsBu"
    "B9tsLziF7UZiefehGkgg5X6iKiBJI/t9FBN+UKfSsZWS2QYlxyBCPU97UOipl8w1LTHvUhn9D4h3wufxmFgrazc7SF+1xZSt"
    "dQOCfnP/I3T8KhshLzrPedQ7lZYQnftWr0jrwNBOf7lhIW9gIQ5xFHdWW6UTdnRERveJjEGDt1yMK1dTB83Tj57NWuF6LEBl"
    "H7Tr4IY0uflh3fbZZ4ryZZ/vfwXnQX2X/JoaG2XkSkK4cmuJ8e3F8awep/bHc2qEq+xTEgIBmQvRm3pHJtOIk0Io/el21RR6"
    "A2NzUmiMWL0jI51+zZ2HK5LZeSqof3SfVHJBriOTXNHieUzDTeE1FxkVvzhldw7KLnqrw+Mip5V6919RNS+oelckmyR1PXHh"
    "NBTuLixGDesErO9Zs039UdFSF3R0z5sdEVab1XXZoUyIyoyi96ayVUusBP6DlSXmJ3K5kdaCLiUoO77OqMxEFMz0ku1w9mAu"
    "YPyKWhAXy+KaqlAcsGxAtc1R3cEaKsVNqdWu0LSZeuCWTZohAMqim77T2USr0ohYwr7Hp5TqYK/TN4t5dcURJN0X6QDjEyti"
    "VjHxZ7EDg7hdCX4Qb4zt64rFKOj3iFoUUDLiQI0BVgfsgYIrlSpdgaC4SVl3X6cvF4nWLPMf0Cfb/gLqWQe9IzH1xysv94cb"
    "N1dbPeZCzv9eIoGXemYza+qNOqwNyixLqPVwIUZRzSQ5DVBbrV2hBv8yBf8LepGCP6JExOyV/a7rqz5teejX6MrT+N9BKZsj"
    "fgF/efbVyNuzlesN6ac47lDbQr8weP7Lj6g2ISev6PTDkijoC4ZfSH1/cjoYQYcodqicoUhYCorO407z1zpp8Bp7yJeC5sNc"
    "TBFBCftCTiuOGSJ8rLNxU1ZHHYbpfgJoPcF7MZDqQzLNrxKu+dVI3NR4w0is2IAn0xsrGe6NRawWnRGroVy8/wV077ItEOWn"
    "M6VrUgZ6ilEnii7MgFhCVsSv9XJQYqSoK+bEX2IqBHtBsImEIUqxJLTxFhzdEpFVKUbAFTIKBTAVLP25xKqPk3qx0LioFz4r"
    "dDUkWnk7VMySOCY+K3lb62NbedZkkOipO5Kg/itsrY+iF9cGMAHrGx1FnGzVnUpioDbOpUc1tdNtRQ2Vi6JStJgWLoRMpGpn"
    "faImiltRE8VWYQmPe5pXuyEzJPFMNNjaubEu8SIXJS45KVM9zOaTWxGXyp/RPkzjGe9ZwncUnXgl8xvEcmsJVTAN+bip5YhV"
    "/07jjso9ND31TNMnZm8B6vmpqQth5mdKGHFABrM+wgwZK1pFv6CW19QuJrRnuM265HEAQjFjc3rWf2JktLkI8fT5jmS+C5ow"
    "duSxrHZBFUcrluve0I8XpSbAxQILSEuHOubf/OevSKT7i1cP1hlhrguJgSDBgjqCzCy9iOTrqj4mBr6lARiXWTCl9uiLYSWR"
    "R0atq+eIBhsgWtgL69FzB4ZaPKPTheGoViyjk+q6FObysqxgfUAVIr9cRK5HOAxh0a4OmJfCyHg4g0DMwTr32s96S+NmjClI"
    "RV5iLARpGjlK9TRqxWkavbL3vHrV12PZGaEhyW6iwrfCemW0wuoNHCEnOd4ifxr50WUFCA/A78QPqA5RFHK4k6JQnBKxvOHM"
    "kd6h4jAI7S6PcPaKbYRoAh2iqQmOFzevRssbx7vjTyZvQAwDwTLoCoUkBOGRY+Ifpn/CUsIhIW5/wVKAcZBYXBEfD7R7jCMC"
    "swT3pZOAke9eyOnfS3DCdu9PtaaFh8QBcPbWngahlte+KD929Kn5in7u280EP37VJ5r6fwaTou3yA4aY1USdtVsN08J30GD9"
    "4Dw1WGnh92qwQ7rlLNZuwNYqxYwcBYwnq6PBOgAR113N6XmXlOGJciBSeHYa7B6vMFNYXqqwVNK49dkL91THG4pnbadmNPG7"
    "d//TaBenhCncP3SUJw4+zmbce/G+6HA54/pqjnRtMjixvk0Wm56Om1bllZ478FJ0dry0OeDxeF00Odbk2v2UXWLv7H0uSu0B"
    "j1yuZcsjcRKTIZS+0lKelEEQgV1V9z+Ok1BB65ieibqfnBkkh3iZKHlQ41ZP8darS3zPGdaiLxSEnUSPFT/DMhqHGV8Ov4ue"
    "UhPfjdfi0xtKD8+2VP0kgfBgRXGPnySY9YoAf6JdzJd+QBHXcGsZ7wqEQsfG9CdqFZ8vDyNyCeDtjM0uSU8U8ICLKngnXeqs"
    "daEJ6j1TX9xBjTXQ/R+YoSReZTb+d2tk5zpOKn4J7Y5bTMePLS3dxJnt4/3MbHNJ46jzxC29S6GonqYWOqmdoGuywgEuv4VH"
    "NbIgHnBRB53Jf7rkImWlwtu2ulufU0JZszoLUKGrmARV8ZArcoOvLDgPg212fNecl6NNM7lriRuiTnjyHfooJApu6xZiPR1V"
    "BXt3uDu4rYv7cmahpN6co9mcaR/DgrKZbg2QPgLq+8tle0Vnv71uvakoZRs43TRQ9wEKqsjYYyL7OmIbXNJhlvalaXYh1PtM"
    "m7uPTiP7VRKl4sen1LFXRBRqxbeyoVbIkatWDFOT+2NamT7/SyzIw9LNINsgJNs+a2D00W9UFXaiXvIiqBe9RvIFb2SDgdEC"
    "LcLFe0/JyrbuWyMPSmdIqr3xSOfk/EXQb45yjlphkvrGCbuvrqrDdVPf9FYYZk0zXprmvVBrfOygac9ojuh7SN2my1RgKmdo"
    "49HOYs9Vnb6j0dAzmfsQ6D189bPXgfjp09ghI33ozQ0JxRoBQxtWJlENEigZbgRkj7RHAiRHUtc4N/d2ItWXILK2T9CUGN9+"
    "rzNwNE1URNStr6DQo0H5PXc3jaubQW+8jFhjccpz4cFc8e2h2EmfFSiX93+B3w5ZUKKVJVYGuJNLUbeBQ3sWGu/54l3NcSVG"
    "i7lgKMMskNr3lUw5zXEH46B3noIGKd4Ku7REFSbm61vRTw2OLL+9v9Iq7KFv6S7sYROVVZe8gvp48/u2bl9saW0vsDRUBbff"
    "gYOO+AWa/eyKO8ukyg9oWTUcPsacYrvPL8NQ38ca2kHTIKfHWgu6vogpCcCY69VA0iKUR0HMSSo/Zmy2HoCdvg0GqD6Z5Mej"
    "AmVETv9MyQkSBlrVQP5lRvvjjm6UEF+/pw+RKY9ESBIE2e2HfpMB1QRSAZtf5cnrggXej4KrlmPMde81HodUYSbH0AdeNkD8"
    "wjeO+ukxaPWZ9j0InVSZ9YPz55ygYR2stY9/kEZ7BOvs9sX+IBsk5Vrba9NBRCc5Xf/3VZW3UVMvpFpNLyhFR8NGp+H0HnkI"
    "RByF4zeOKvPRPwencf4hNiZ1+AknVbqdQszP65lK6IfRSkZDgiq6fTq/Ig/r6RyLfvToirEr9f4T3pqle05ZU19NJgls1bt7"
    "TZ9oNSFAJhNQf2kxB68JKy/V427BOm3NGNpfi4EkI7nRwoAJ09IXtgyAkRj0bHIBm29wYY/303MHzojPfmOjt4HUcWZXw39p"
    "NW8x4bfEzj+4z6g7OWUA43lcimlQi701XNCuG3o0mNoPfcK9fJWXI+Kw8PYm3+OHLtUWp7t3BqoFXA4qPUol3l5B7LyCQv+U"
    "nuUhfHH5DJ5eldT+sH0GV0o/lZoSAYkc9z9Wecbf26avpZ8FIcLBACCpVqUAAT5/Ma/Z3iClzWbtV5HUSNW9DLqXt93htu0N"
    "JES2h4UdbBvySxy4wUV7fJmMAGfJ/c/0Ds8N7Tt20hAZXguQbwWYN1olffxatPElNzsV95jVkY/uTbGZRLrDHkchIurm96DA"
    "3hJT8Wj1uJkDv4SPzS9jrx+CI6+QC/WODDf5YUe8s8X6kFe0N7BR31z+rSoFxGKYvw91A/KqHv6BLyjSACBUC6rjIYjWoiKd"
    "cXglVEJUjhO6J/o6ok6uc1Lp6YlPGReJ1Y2RxHKebkGR49ChU2YIETbJVMWmzTWNHfgneo7zhssybdenTijNPOQzSg6eeZgz"
    "Jnclcg9+m633h4rO8UT7+gFSSMECxr4m+CgMCNIb0c6lZZ7XPiXoJW/qvilJ00GFuzan/w43EeWCFE7Qjw5/76EHJ+fbEMRP"
    "8bnwk3FNzjzPmy5Hy8OdVP3o4FloKzFCUqltk+gKEc5pNi5e1E24umyh8++wQLLDa/oTS0gsxbvgWjq76CnJk2itRyZ6qjtn"
    "ef9ztcR9Ax/hVUg7jnVmh8JpWHLy//1PqufEOhu+YaaF5YehsOI0OXTpa0mOs+E3oqtd/YmqAM3TiVWu4PNjzgceHRw2hAjb"
    "LKQfm9bMgfd6o3nEMvPnUlX6uAX/V26vwO23EksJtn0BS7n9iCUS0P/W8M38KL75Vgew7sxfa5MK2UyVtqYYtnuIwond6qEv"
    "eg6RoxBpVNouCqwya8Wj1Wrxa4cDJj1PQaJtOkaGNPaRm8KhnuQ0SVd4ODX0G2f0slDHtfPrN3kHq7kTqRfP5gHhLuwjeIvd"
    "G9e/FmOPc09XNSmwb66GUGRP7P6I643YU7ju1A4QDWZf/bp+fpg52CROofQnExRGSKxcU8OMeg9d0uYHO6QqkADEJpGjCmmZ"
    "C4CImX77xVQdozr7NbHu/yQyBPX9r5fA7l26D0BtFEgVHmt39gBtzVCHI8Ap3P5kJP+dasVjRdDdAa5Q0yU0sOeF6Skbu9Z+"
    "DbLReV9GyXu9dGA6oEGnxREuilHwWq/EI/og/OCciE9+iOyaqqoILwQa2zvpgrgz/AwdN0R0OjeERALtfWHwb7I7kDtvhUsE"
    "kxErYea/aV0BqmfiNfXNTpR3cDun9R9cV/fj3BNHIUsei2E0+ViZk9sicGDB8Fy0ze9zj1J0oV4k+D+zilbX4oinek8dNzxd"
    "How4T7yqoawkALXI95bvYSXepQIzVG4mpyBZ45O4cPPMBzMK5zd+8ZDki/XGQZJ6jBFzHCqtb55Dgj3YUptzPkkduCo6u4Ot"
    "8SKqKp845hD2XnAGZgXZD7JGqGG554Kabh/YqP13osfhPgcfuLja+V0BQOEHZpA20Q4XXgn4960X0d8LpBDmIGtFm79VQeJJ"
    "Aw22yYdhFX9xjyl8EsRwyh6U2Ku6vGnsIivj8/HXYzLED1LnNdxE9YH0DlipPoUotE67dd+JDUql1FVXhuj8bpNp52HHbJFi"
    "RjWcYtGZTBb8d3TPu+nRozDCuTq4OPvmbTr2kGsephxz4RsQWIE9AIm2f0QegaoA+O7Kwk7+7LTlwpXCZqIaGeV/Ns0GMNVI"
    "NGcQ+StgG02gM0M4gayVaILJ39Fk1i2RVxJ8JNXuYE7hkBW7hbwlAQovAE0wPIfA8F9RY3sS96QcqY0nAAM5GQ6i0kCUs+sH"
    "1jOMvuDTYl3wsiUiuWLEevhqtFqnXv1kgAWdcmNPyX0jKntflutd1tZajvEBLHJV/pDt8BzLPh/WVJ0kgPr01Ur2NgD1CRpz"
    "V5kMVsIQVEeg30dV115ei5PAU38LLju1Z80zmvcD26W6KSFDyXM4jtDCw+xtk8mvFVb91gBFmwYz8D5CL/gqu1xTmwssUS4e"
    "X5e7Ah9ejOtl7YqYFJMPQemEHa+JC8NH5MKTlcRSTy3qPyLulWF3XyrjfFffxOo5u+D/tDa4kpSS4uMZEUR1rRLHI6uZI47q"
    "Uge+VWn3mTvCA4g4nFF2FFg/pH6F2E8dmCI6W6ZQpBmsyqZcZVRcC06qHw6oj0OSeUz/mwNTLHgbjkaqmf14AtENth0Vfyz2"
    "ucITWAwNStBm1OcAPdk3aCVSfw7U72gcHnk7PPJu8s/vPCgv+DPNuW5h2VXzjk3nghoOOhVSrHI+Rh8ntomfnG2OdF7j78de"
    "VVuDe6J3mPLrA/v4eD8uCHsyrUBfRJcXLiIaj14JYnArc7nQTxrX+1QXRzVclcBLwnKtdtVqcr6wSpo9sBFb+jc54yQl0XHY"
    "8FUYx90tQ55JzkELsrHLV9DMOtt+Fl6nFQYFUWXOCxTXVbYb0mhKuj1OgyOGi/5jhh5UWEe4yidrfAhRT9TC/g1bciI2ikG/"
    "YuPbmxv7ZnPj8WAZtcXv6WjsGx2N/aGOxsQM82exyo7hCDxAt9A7xEmxzdZXGXFNDyMENopc0CgPq0VCdEW9C2+jk37Ad6nl"
    "1uPYFRm6D38cGifraExMkz71CTK2KSV5hmR9xxms+/JTtqMkcTySyaSqraddRjXuM/HSWOMJp+I1niONjnSq1fgb7lKMA6gX"
    "ulVqweOOwvMQzNw6Jys4sRXjprrwkp8HeG/x4qyn2lUCPpLZO3KO3IEOOJs4e35gnPSqYJ2CmbjHOXDNDN+oTZVoYr9+MOja"
    "YQAzt8tMkKz/ZtR1UaC5kZHxCELqZBYS0L4CD2ypK7loM0NZ0YAoz3uZxUhVMRRRTFH4pX2I0UytWwoEyGO25IHf7WMzCpDo"
    "mNKCOKEq6lbm4LkECcRCSYRsMfIOKQo/5LiJZSNWLAuANV1RYoCPyJQmI+RIW7SHRApYlz0mUOOGQSNUQ4FEG/AZGFRQNgkd"
    "ND40LBMmiHm892u18RTXkURcc8wLy9VVoTkhpowi3I/jSbUdP3jpNrZca1SAuHADLH5UW6qG10KzV023IGqr0nrV2Op11CZV"
    "sCFSQ/6cBgdK6jChn8zh4ofnajxj9ApSN66hFDdIb+i0gYEp8KoA7YHwkFJVol6XbykGdf/rWnR/WBbbZSk6HK9sbTiUpRYT"
    "YIqHfP3DsPuXxmBKhPDJ+6hNwP9nZtCgU6ixQQmXiDro0JVtm3orgPCe334crGUiGeisItui0CqzDrwUnbP0e6Az5vd/r/pj"
    "Ei77UOxgq28EfMqgh5JcAE5MhmSW1RMTm56YMQBZ0cP7YGLDBxO7+GD8+FzPlG5DT8guAYkFS4QcADmZbTdPWLKP97/uIQrX"
    "BCbXMDdNIIeM0YrkxJky74MNbaXfZjzVcnsG7NlYjsBUCbuRHR5vZJtYOdjXBj58gDI24pOMhPv//4ObqfNRAQA="
)

if not RUTA_DATOS.exists():
    RUTA_DATOS.parent.mkdir(parents=True, exist_ok=True)
    RUTA_DATOS.write_bytes(gzip.decompress(base64.b64decode(_BLOB)))
    print(f"Corpus escrito en {RUTA_DATOS} (copia embebida).")
else:
    print(f"Se usará el corpus existente en {RUTA_DATOS}.")

sha_real = hashlib.sha256(RUTA_DATOS.read_bytes()).hexdigest()
print("SHA-256:", sha_real[:16], "…")
print("Coincide con la versión embebida:", sha_real == SHA_ESPERADO)

### 4.2 · Carga y particiones

Las particiones vienen **fijadas en el archivo** (campo `split`), no se
calculan aquí. Es una decisión deliberada: si cada notebook hiciera su propio
`train_test_split`, cuatro arquitecturas estarían evaluándose sobre conjuntos
distintos y las métricas no serían comparables entre sí.

La partición es estratificada por categoría (12 entrenamiento + 3 validación
por clase), generada con semilla 42.

In [ ]:
import json
from collections import Counter

registros = [json.loads(l) for l in RUTA_DATOS.read_text(encoding="utf-8").splitlines()]

train = [r for r in registros if r["split"] == "train"]
val   = [r for r in registros if r["split"] == "validation"]
demo  = [r for r in registros if r["es_demo"]]

CATEGORIAS = [
    "suma", "resta", "multiplicacion", "division", "operaciones_combinadas",
    "potencias_raices", "fracciones", "porcentajes", "ecuaciones",
    "geometria", "estadistica_probabilidad",
]
CAT2ID = {c: i for i, c in enumerate(CATEGORIAS)}
ID2CAT = {i: c for c, i in CAT2ID.items()}

print(f"Total: {len(registros)}  |  train: {len(train)}  |  validación: {len(val)}")
print(f"Ejemplos de demostración (todos en validación): {[d['id'] for d in demo]}")
print()
print("Distribución por categoría (train / val):")
ctr, cva = Counter(r["categoria"] for r in train), Counter(r["categoria"] for r in val)
for c in CATEGORIAS:
    print(f"  {c:26s} {ctr[c]:3d} / {cva[c]:2d}")
print()
print("Ejemplo completo:")
print(json.dumps(train[0], ensure_ascii=False, indent=2))

Un ejemplo del corpus se ve así:

```
entrada : "María tiene 48 caramelos y quiere repartirlos por igual entre 6
           amigos. ¿Cuántos caramelos recibirá cada amigo?"
salida  : "Paso 1: Repartir en partes iguales es dividir.
           Paso 2: 48 ÷ 6 = 8.
           Respuesta final: 8 caramelos"
valor   : "8"
```

El campo `valor` es la clave de toda la evaluación automática: es la respuesta
en forma canónica (sin unidades ni texto). Comparar `valor` contra lo que el
modelo escribe después de `Respuesta final:` nos da una métrica objetiva de
**si el modelo resolvió bien el problema**, independiente de cómo lo redactó.

### Métricas de generación

Antes de tocar el modelo definimos **cómo vamos a medirlo**. Fijar las métricas
antes de ver resultados evita el sesgo de elegir después la métrica que mejor
nos deja.

| Métrica | Qué mide | Por qué está |
|---|---|---|
| **Exactitud de la respuesta** | ¿El número final es correcto? | Es la única métrica que responde "¿sirve como tutor?". Un procedimiento bonito con resultado equivocado es un fracaso. |
| **Formato válido** | ¿Usó `Paso N:` y `Respuesta final:`? | Separa *aprender a resolver* de *aprender a formatear*. El fine-tuning con pocos datos suele mejorar mucho lo segundo y poco lo primero; sin esta métrica confundiríamos ambos efectos. |
| **ROUGE-L** | Solapamiento de la explicación con la de referencia | Aproxima si el procedimiento se parece al esperado. Es una métrica débil (premia coincidencia léxica, no razonamiento) y así hay que leerla. |
| **Longitud media** | Palabras generadas | Detecta el modo de fallo típico del baseline: divagar o repetirse hasta agotar `max_new_tokens`. |

Nota metodológica importante: para extraer la respuesta del modelo usamos el
marcador `Respuesta final:` y, si no aparece, **el último número del texto**.
Sin ese respaldo estaríamos castigando al baseline por desconocer un formato
que todavía no le hemos enseñado, y la mejora del fine-tuning se vería
artificialmente enorme.

In [ ]:
import re
import unicodedata
from fractions import Fraction

MARCADOR_RESPUESTA = "Respuesta final:"

_PAT_RESPUESTA = re.compile(r"Respuesta\s+final\s*:\s*(.+)", re.IGNORECASE)
_PAT_PASO1     = re.compile(r"Paso\s*1\s*:", re.IGNORECASE)
_PAT_CATEGORIA = re.compile(r"Categor[ií]a\s*:\s*([a-zA-Z_]+)", re.IGNORECASE)
# La alternativa de fracción va primero: en "3/5" queremos capturar la fracción
# completa, no el "3" suelto.
_PAT_VALOR = re.compile(r"-?\d+(?:\.\d+)?\s*/\s*-?\d+(?:\.\d+)?|-?\d+(?:\.\d+)?")


def _sin_miles(texto):
    """Quita la coma como separador de miles. El corpus usa el punto como
    separador decimal y nunca la coma, así que la conversión no es ambigua."""
    return texto.replace(",", "")


def a_float(valor):
    if valor is None:
        return None
    try:
        return float(Fraction(valor)) if "/" in valor else float(valor)
    except (ValueError, ZeroDivisionError):
        return None


def valor_predicho(texto):
    """Extrae la respuesta del modelo en forma canónica.

    Prioridad 1: el número que sigue al marcador 'Respuesta final:'.
    Prioridad 2: el último número del texto.

    El segundo caso importa para que la comparación sea JUSTA: un modelo sin
    fine-tuning no conoce nuestro formato, y penalizarlo por eso mediría
    obediencia al formato, no capacidad matemática. Con el fallback medimos lo
    segundo; el apego al formato se mide aparte con `formato_valido`.
    """
    m = _PAT_RESPUESTA.search(texto)
    if m:
        linea = m.group(1).splitlines()[0]
        v = _PAT_VALOR.search(_sin_miles(linea))
        if v:
            return v.group(0).replace(" ", "")
    todos = _PAT_VALOR.findall(_sin_miles(texto))
    return todos[-1].replace(" ", "") if todos else None


def respuesta_correcta(generado, valor_oro, tol=1e-6):
    a = a_float(valor_predicho(generado))
    b = a_float(valor_oro)
    if a is None or b is None:
        return False
    return abs(a - b) <= tol * max(1.0, abs(b))


def formato_valido(texto):
    """¿El modelo produjo la estructura que le enseñamos?"""
    return bool(_PAT_PASO1.search(texto)) and bool(_PAT_RESPUESTA.search(texto))


def categoria_predicha(texto):
    m = _PAT_CATEGORIA.search(texto)
    return m.group(1).lower() if m else None


def _tokens(texto):
    t = unicodedata.normalize("NFKD", texto.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", t)


def _lcs(a, b):
    """Longitud de la subsecuencia común más larga (programación dinámica)."""
    previa = [0] * (len(b) + 1)
    for x in a:
        actual = [0]
        for j, y in enumerate(b):
            actual.append(previa[j] + 1 if x == y else max(previa[j + 1], actual[j]))
        previa = actual
    return previa[-1]


def rouge_l(generado, referencia):
    """ROUGE-L (F1 sobre la subsecuencia común más larga).

    Se implementa a mano en lugar de usar `evaluate` para que el notebook no
    dependa de descargas en tiempo de ejecución y el número sea exactamente
    reproducible.
    """
    p, r = _tokens(generado), _tokens(referencia)
    if not p or not r:
        return 0.0
    l = _lcs(p, r)
    if l == 0:
        return 0.0
    prec, rec = l / len(p), l / len(r)
    return 2 * prec * rec / (prec + rec)


def evaluar_generacion(generados, registros):
    """Métricas de generación sobre un conjunto de ejemplos.

    exactitud       : ¿la respuesta final es numéricamente correcta?  <- la que importa
    formato_valido  : ¿respetó la estructura Paso N / Respuesta final?
    rouge_l         : ¿se parece el procedimiento al de referencia?
    long_media      : longitud media en palabras (detecta divagación)
    """
    assert len(generados) == len(registros)
    n = len(generados)
    correctas = [respuesta_correcta(g, r["valor"]) for g, r in zip(generados, registros)]
    formatos  = [formato_valido(g) for g in generados]
    rouges    = [rouge_l(g, r["salida"]) for g, r in zip(generados, registros)]
    return {
        "n": n,
        "exactitud": sum(correctas) / n,
        "formato_valido": sum(formatos) / n,
        "rouge_l": sum(rouges) / n,
        "long_media": sum(len(g.split()) for g in generados) / n,
        "_correctas": correctas,
    }


def tabla_metricas(antes, despues, titulo="Baseline vs Fine-tuned"):
    """Imprime la comparación en el formato que usaremos en el informe."""
    filas = [
        ("Exactitud de la respuesta", "exactitud", "{:.1%}"),
        ("Formato válido",            "formato_valido", "{:.1%}"),
        ("ROUGE-L del procedimiento", "rouge_l", "{:.3f}"),
        ("Longitud media (palabras)", "long_media", "{:.1f}"),
    ]
    ancho = 30
    print(titulo)
    print("=" * 68)
    print(f"{'Métrica':{ancho}s} {'Baseline':>12s} {'Fine-tuned':>12s} {'Δ':>10s}")
    print("-" * 68)
    for etiqueta, clave, fmt in filas:
        a, d = antes[clave], despues[clave]
        print(f"{etiqueta:{ancho}s} {fmt.format(a):>12s} {fmt.format(d):>12s} "
              f"{d - a:>+10.3f}")
    print("=" * 68)

In [ ]:
import json
from pathlib import Path

DIR_RESULTADOS = Path("resultados")
DIR_RESULTADOS.mkdir(exist_ok=True)


def guardar_resultados(nombre, payload):
    """Persiste las métricas para que el notebook de comparación las agregue.

    Sin este paso, comparar las cuatro arquitecturas obligaría a re-ejecutar
    todo en una sola sesión. Con él, cada notebook se ejecuta cuando se pueda y
    la comparación se hace al final leyendo los JSON.
    """
    limpio = {}
    for k, v in payload.items():
        if isinstance(v, dict):
            limpio[k] = {kk: vv for kk, vv in v.items() if not kk.startswith("_")}
        elif not k.startswith("_"):
            limpio[k] = v
    ruta = DIR_RESULTADOS / f"{nombre}.json"
    ruta.write_text(json.dumps(limpio, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Métricas guardadas en {ruta}")
    return ruta

## 5 · Preprocesamiento

### Estrategias didácticas por categoría

Esta es la pieza de diseño central del notebook: qué hacemos realmente con la
categoría una vez que la conocemos.

La opción ingenua sería inyectar la etiqueta en el prompt ("Este es un problema
de porcentajes"). Aporta poco: el modelo ya puede inferirlo del enunciado.

La opción útil es asociar a cada categoría **el procedimiento que un profesor
enseñaría** para ese tipo de problema. La categoría deja de ser una etiqueta y
pasa a ser una clave para recuperar conocimiento pedagógico que el modelo no
tiene por qué haber aprendido de 132 ejemplos.

Esto convierte al clasificador en algo parecido a un recuperador: enruta hacia
la instrucción especializada correcta. Es la misma lógica de un sistema RAG,
con un espacio de recuperación de 11 elementos en lugar de un índice vectorial.

In [ ]:
ESTRATEGIAS = {
    "suma": "Identifica las cantidades que se juntan y súmalas. Verifica que las unidades coincidan y alinea los decimales si los hay.",
    "resta": "Identifica la cantidad inicial y la que se quita o se compara. Recuerda que el resultado puede ser negativo.",
    "multiplicacion": "Identifica el valor unitario y cuántas veces se repite. Multiplica. Si hay decimales, cuenta las cifras decimales del resultado.",
    "division": "Identifica el total y en cuántas partes se reparte. Divide. Si el contexto no admite fracciones (personas, buses, cajas), redondea y explica por qué.",
    "operaciones_combinadas": "Aplica la jerarquía de operaciones: primero los paréntesis, después multiplicaciones y divisiones de izquierda a derecha, y por último sumas y restas. Resuelve una operación por paso.",
    "potencias_raices": "Calcula primero las potencias y raíces, y solo después las demás operaciones. Explica qué significa la potencia o la raíz que estás calculando.",
    "fracciones": "Si sumas o restas, halla el denominador común. Si multiplicas, multiplica numeradores y denominadores. Si divides, multiplica por la fracción inversa. Simplifica el resultado.",
    "porcentajes": "Convierte el porcentaje a decimal dividiendo entre 100 y multiplica. Si piden un descuento o un aumento, calcula primero la parte y después súmala o réstala al valor original.",
    "ecuaciones": "Plantea la ecuación con una incógnita. Despeja aplicando la misma operación a ambos lados hasta dejar la incógnita sola.",
    "geometria": "Escribe explícitamente la fórmula que vas a usar antes de sustituir los valores. Indica las unidades del resultado (lineales, cuadradas o cúbicas).",
    "estadistica_probabilidad": "Para un promedio, suma los valores y divide entre cuántos son. Para una probabilidad, cuenta los casos favorables y divídelos entre los casos posibles.",
}

assert set(ESTRATEGIAS) == set(CATEGORIAS), "Falta una estrategia para alguna categoría"
for cat, est in list(ESTRATEGIAS.items())[:3]:
    print(f"[{cat}]\n  {est}\n")

In [ ]:
INSTRUCCION = (
    "Eres un tutor de matemáticas. Resuelve el siguiente problema explicando "
    "el procedimiento paso a paso y termina con la respuesta final."
)


def construir_prompt(entrada, categoria=None):
    """Prompt de generación, con o sin condicionamiento por categoría.

    Sin categoría es EXACTAMENTE el prompt del notebook 1: así la comparación
    A vs B aísla el efecto del enrutamiento y nada más.
    """
    sistema = INSTRUCCION
    if categoria is not None:
        sistema += (
            f"\nEste problema es de tipo '{categoria}'. "
            f"Estrategia recomendada: {ESTRATEGIAS[categoria]}"
        )
    mensajes = [
        {"role": "system", "content": sistema},
        {"role": "user", "content": entrada},
    ]
    return tokenizer_qwen.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )

## 6 · Tokenización y carga de los modelos

Cada etapa conserva su propio tokenizador. No son intercambiables: BETO usa
WordPiece con 31k tokens, Qwen usa BPE con 151k. Un `input_ids` de uno no
significa nada para el otro. En un pipeline con dos tokenizadores, mezclarlos
es un error silencioso —no lanza excepción, simplemente produce basura—, así
que los nombramos de forma inequívoca: `tokenizer_bert` y `tokenizer_qwen`.

In [ ]:
import torch
from transformers import (AutoModelForCausalLM, AutoModelForSequenceClassification,
                          AutoTokenizer)
from peft import PeftModel

# ---- Etapa 1: clasificador
tokenizer_bert = AutoTokenizer.from_pretrained(DIR_BERT)
modelo_bert = AutoModelForSequenceClassification.from_pretrained(DIR_BERT).to(DEVICE)
modelo_bert.eval()
print(f"Etapa 1 cargada: {modelo_bert.config._name_or_path} "
      f"({modelo_bert.config.num_labels} clases)")

# ---- Etapa 2: generador
tokenizer_qwen = AutoTokenizer.from_pretrained(DIR_ADAPTADOR)
if tokenizer_qwen.pad_token is None:
    tokenizer_qwen.pad_token = tokenizer_qwen.eos_token

base = AutoModelForCausalLM.from_pretrained(MODELO_QWEN, torch_dtype=torch.float16).to(DEVICE)
modelo_qwen = PeftModel.from_pretrained(base, DIR_ADAPTADOR)
# Fusionamos los pesos LoRA en el modelo base: en inferencia elimina la
# indirección del adaptador y acelera la generación entre un 5 y un 10%.
modelo_qwen = modelo_qwen.merge_and_unload()
modelo_qwen.eval()
print(f"Etapa 2 cargada: {MODELO_QWEN} + adaptador LoRA (fusionado)")

In [ ]:
# El orden de las etiquetas del clasificador debe coincidir con CATEGORIAS.
# Si no coincidiera, el pipeline enrutaría con estrategias equivocadas sin dar
# ningún error visible: es el fallo silencioso más peligroso de este diseño.
etiquetas_modelo = [modelo_bert.config.id2label[i] for i in range(modelo_bert.config.num_labels)]
assert etiquetas_modelo == CATEGORIAS, (
    f"Desalineación de etiquetas.\n  modelo: {etiquetas_modelo}\n  corpus: {CATEGORIAS}"
)
print("Etiquetas alineadas correctamente entre el clasificador y el corpus.")

## 7 · Baseline

El baseline de este notebook **no es el modelo sin entrenar**: eso ya se midió
en el notebook 1. Aquí la pregunta es otra —¿aporta algo el enrutamiento?— y
por tanto el punto de comparación correcto es **Qwen fine-tuned trabajando
solo, sin categoría** (configuración A).

Elegir mal el baseline es el error metodológico más frecuente al evaluar
sistemas compuestos: comparar el pipeline completo contra un modelo sin
entrenar demuestra que el fine-tuning funcionó, no que la arquitectura modular
aporte nada.

In [ ]:
from tqdm.auto import tqdm


@torch.no_grad()
def clasificar(textos, batch=16):
    """Etapa 1. Devuelve (categoría, confianza) para cada texto."""
    salida = []
    for i in range(0, len(textos), batch):
        lote = tokenizer_bert(textos[i:i + batch], padding=True, truncation=True,
                              max_length=LONGITUD_MAX_CLF, return_tensors="pt").to(DEVICE)
        probs = modelo_bert(**lote).logits.softmax(-1)
        conf, idx = probs.max(-1)
        salida.extend(zip([CATEGORIAS[j] for j in idx.cpu().tolist()],
                          conf.cpu().tolist()))
    return salida


@torch.no_grad()
def generar(entradas, categorias=None, max_new_tokens=MAX_TOKENS_GEN):
    """Etapa 2. `categorias=None` genera sin condicionamiento (configuración A)."""
    salidas = []
    for i, entrada in enumerate(tqdm(entradas, desc="generando")):
        cat = categorias[i] if categorias is not None else None
        prompt = construir_prompt(entrada, cat)
        inputs = tokenizer_qwen(prompt, return_tensors="pt",
                                add_special_tokens=False).to(DEVICE)
        out = modelo_qwen.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer_qwen.pad_token_id,
            eos_token_id=tokenizer_qwen.eos_token_id,
        )
        nuevos = out[0][inputs["input_ids"].shape[1]:]
        salidas.append(tokenizer_qwen.decode(nuevos, skip_special_tokens=True).strip())
    return salidas

In [ ]:
# ---- Configuración A: Qwen fine-tuned solo (baseline del pipeline)
gen_A = generar([r["entrada"] for r in val], categorias=None)
metricas_A = evaluar_generacion(gen_A, val)

print()
print("A · Qwen fine-tuned sin enrutamiento")
for k, v in metricas_A.items():
    if not k.startswith("_"):
        print(f"  {k:16s}: {v}")

## 8 · Configuración del pipeline

No hay hiperparámetros de entrenamiento que ajustar aquí, porque **este
notebook no entrena nada nuevo**: compone dos modelos ya entrenados. Lo que sí
hay que configurar son las decisiones de diseño del sistema.

### Umbral de confianza

El clasificador devuelve una probabilidad además de la etiqueta. Con ella se
puede decidir *no enrutar* cuando la predicción es dudosa y caer en el prompt
genérico, que es exactamente la configuración A.

Es una salvaguarda barata contra la propagación de error: si el clasificador no
está seguro, el sistema se degrada al comportamiento del modelo único en lugar
de recibir una estrategia equivocada.

Con `UMBRAL_CONFIANZA = 0.0` el mecanismo queda desactivado y se enruta siempre;
así medimos primero el pipeline puro. Después pueden subirlo y volver a evaluar
para ver si conviene.

In [ ]:
UMBRAL_CONFIANZA = 0.0   # 0.0 = enrutar siempre; prueben 0.5 o 0.7 después


def pipeline(entradas, umbral=UMBRAL_CONFIANZA, verbose=False):
    """Sistema completo de dos etapas."""
    clasificaciones = clasificar(entradas)
    categorias, descartadas = [], 0
    for cat, conf in clasificaciones:
        if conf >= umbral:
            categorias.append(cat)
        else:
            categorias.append(None)     # se cae al prompt genérico
            descartadas += 1
    if verbose and descartadas:
        print(f"{descartadas} predicciones por debajo del umbral: se usó prompt genérico.")
    generaciones = generar(entradas, categorias=categorias)
    return generaciones, clasificaciones, categorias

### Demostración del flujo con un solo ejemplo

Antes de evaluar en lote, conviene ver el sistema funcionando paso a paso.

In [ ]:
ejemplo = demo[0]
print("USUARIO:", ejemplo["entrada"])
print()

cat, conf = clasificar([ejemplo["entrada"]])[0]
print(f"ETAPA 1 · BETO clasifica: '{cat}' (confianza {conf:.1%})")
print(f"          categoría real: '{ejemplo['categoria']}'  "
      f"-> {'correcta' if cat == ejemplo['categoria'] else 'INCORRECTA'}")
print()
print(f"SELECCIÓN · estrategia asociada:")
print(f"          {ESTRATEGIAS[cat]}")
print()
salida = generar([ejemplo["entrada"]], categorias=[cat])[0]
print("ETAPA 2 · Qwen genera:")
print(salida)
print()
print(f"Referencia: {ejemplo['salida']}")

## 9 · Ejecución del pipeline

Ejecutamos las configuraciones B (pipeline real) y C (oráculo) sobre los mismos
33 ejemplos de validación.

La configuración C es un **experimento de ablación**: sustituye el clasificador
por sus etiquetas verdaderas. No es un sistema desplegable —en producción no
existe la etiqueta real—, pero es la única forma de separar "el enrutamiento no
sirve" de "el clasificador no acierta lo suficiente". Son dos diagnósticos con
soluciones completamente distintas.

In [ ]:
# ---- Configuración B: pipeline real (categoría predicha)
gen_B, clasificaciones, cats_predichas = pipeline([r["entrada"] for r in val], verbose=True)
metricas_B = evaluar_generacion(gen_B, val)

aciertos_clf = [c == r["categoria"] for (c, _), r in zip(clasificaciones, val)]
accuracy_clf = sum(aciertos_clf) / len(aciertos_clf)

print()
print("B · Pipeline BERT -> Qwen")
print(f"  accuracy del clasificador: {accuracy_clf:.1%}")
for k, v in metricas_B.items():
    if not k.startswith("_"):
        print(f"  {k:16s}: {v}")

In [ ]:
# ---- Configuración C: oráculo (categoría real)
gen_C = generar([r["entrada"] for r in val], categorias=[r["categoria"] for r in val])
metricas_C = evaluar_generacion(gen_C, val)

print()
print("C · Oráculo (categoría real, techo del pipeline)")
for k, v in metricas_C.items():
    if not k.startswith("_"):
        print(f"  {k:16s}: {v}")

### Weights & Biases

W&B registra automáticamente la curva de pérdida, los hiperparámetros y el
consumo de GPU. Es lo que después nos permitirá comparar las cuatro
arquitecturas sobre los mismos ejes en lugar de sobre capturas de pantalla.

Convención de nombres del proyecto (idéntica en los cinco notebooks):

- **Proyecto:** `tutor-matematicas-arquitecturas`
- **Run:** `pipeline-qwen-bert`
- **Tags:** identifican arquitectura y fase, para poder filtrar en el panel

Si no quieren usar W&B, pongan `USAR_WANDB = False`: el notebook seguirá
funcionando y las métricas se guardarán igual en `resultados/`.

In [ ]:
import os

USAR_WANDB   = True          # ponlo en False para trabajar sin conexión a W&B
PROYECTO     = "tutor-matematicas-arquitecturas"
NOMBRE_RUN   = "pipeline-qwen-bert"
TAGS         = ["pipeline", "qwen", "bert", "modular", "cascada"]

if USAR_WANDB:
    import wandb
    wandb.login()            # pedirá la API key la primera vez
    os.environ["WANDB_PROJECT"] = PROYECTO
    os.environ["WANDB_LOG_MODEL"] = "false"
    REPORTAR_A = "wandb"
else:
    os.environ["WANDB_MODE"] = "disabled"
    REPORTAR_A = "none"

print(f"Registro de experimentos: {REPORTAR_A}  |  run: {NOMBRE_RUN}")

## 10 · Integración con Weights & Biases

Este notebook no lanza un `Trainer`, así que no hay curvas de pérdida que
registrar: el registro es puramente de evaluación. Aun así usa el mismo
proyecto y la misma convención de nombres, para que en el panel de W&B las
cuatro arquitecturas aparezcan una al lado de la otra.

Se registran las tres configuraciones como métricas resumen, la tabla completa
de generaciones con la categoría predicha y la real, y la latencia por etapa.

## 11 · Evaluación

### Propagación de error

La medición clave del notebook. Separamos los ejemplos según si el clasificador
acertó, y comparamos la exactitud del generador en cada grupo.

Interpretación de lo que puede salir:

- **Exactitud mucho menor cuando el clasificador falla** → el enrutamiento
  tiene efecto real (para bien y para mal). Mejorar el clasificador mejoraría
  el sistema completo.
- **Exactitud parecida en ambos grupos** → la estrategia de categoría apenas
  influye en la generación. El clasificador es decorativo y el pipeline no se
  justifica.

Cuidado con el tamaño de muestra: si el clasificador acierta 29 de 33, el grupo
de errores tiene 4 ejemplos. Cualquier porcentaje sobre 4 ejemplos es
anecdótico. Repórtenlo con el `n` al lado, siempre.

In [ ]:
correctas_B = metricas_B["_correctas"]

grupo_ok  = [c for c, a in zip(correctas_B, aciertos_clf) if a]
grupo_mal = [c for c, a in zip(correctas_B, aciertos_clf) if not a]

print("Propagación de error en la cascada")
print("=" * 66)
print(f"Clasificación CORRECTA   n={len(grupo_ok):2d}  "
      f"exactitud del generador = {sum(grupo_ok)/max(len(grupo_ok),1):.1%}")
print(f"Clasificación INCORRECTA n={len(grupo_mal):2d}  "
      f"exactitud del generador = {sum(grupo_mal)/max(len(grupo_mal),1):.1%}")
print("=" * 66)
if len(grupo_mal) < 5:
    print(f"AVISO: solo {len(grupo_mal)} ejemplos mal clasificados. "
          "La cifra de ese grupo es anecdótica, no una estimación.")

### Latencia por etapa

El argumento práctico. Si la clasificación cuesta el 2% del tiempo total, el
pipeline es esencialmente gratis frente al modelo único y solo hay que
justificarlo por calidad. Si costara el 40%, habría que discutirlo.

In [ ]:
import time

muestra = [r["entrada"] for r in val[:8]]

t0 = time.perf_counter()
_ = clasificar(muestra)
t_clf = (time.perf_counter() - t0) / len(muestra)

t0 = time.perf_counter()
_ = generar(muestra, categorias=[c for c, _ in clasificar(muestra)])
t_gen = (time.perf_counter() - t0) / len(muestra)

print(f"Etapa 1 · clasificación : {t_clf*1000:7.1f} ms/pregunta")
print(f"Etapa 2 · generación    : {t_gen*1000:7.1f} ms/pregunta")
print(f"Total                   : {(t_clf+t_gen)*1000:7.1f} ms/pregunta")
print(f"\nLa clasificación representa el {100*t_clf/(t_clf+t_gen):.1f}% del tiempo total.")

## 12 · Comparación Baseline vs Pipeline

In [ ]:
configs = [
    ("A · Qwen solo (baseline)",      metricas_A),
    ("B · Pipeline BERT -> Qwen",     metricas_B),
    ("C · Oráculo (categoría real)",  metricas_C),
]

print(f"{'Configuración':32s} {'Exactitud':>10s} {'Formato':>9s} {'ROUGE-L':>9s} {'Palabras':>9s}")
print("=" * 74)
for nombre, m in configs:
    print(f"{nombre:32s} {m['exactitud']:>10.1%} {m['formato_valido']:>9.1%} "
          f"{m['rouge_l']:>9.3f} {m['long_media']:>9.1f}")
print("=" * 74)
print()
print(f"Beneficio del enrutamiento perfecto  (C - A): "
      f"{metricas_C['exactitud'] - metricas_A['exactitud']:+.1%}")
print(f"Costo de los errores del clasificador (C - B): "
      f"{metricas_C['exactitud'] - metricas_B['exactitud']:+.1%}")
print(f"Aporte neto del pipeline real         (B - A): "
      f"{metricas_B['exactitud'] - metricas_A['exactitud']:+.1%}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

nombres = [c[0] for c in configs]
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(configs)); ancho = 0.38
ax.bar(x - ancho/2, [c[1]["exactitud"] for c in configs], ancho, label="Exactitud")
ax.bar(x + ancho/2, [c[1]["formato_valido"] for c in configs], ancho, label="Formato válido")
ax.set_xticks(x); ax.set_xticklabels(nombres, rotation=12, ha="right", fontsize=9)
ax.set_ylim(0, 1.05); ax.set_ylabel("Proporción")
ax.set_title(f"Pipeline modular · {len(val)} ejemplos de validación")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
print("Comparación cualitativa sobre los ejemplos de demostración")
gen_demo_A = generar([d["entrada"] for d in demo], categorias=None)
cats_demo = [c for c, _ in clasificar([d["entrada"] for d in demo])]
gen_demo_B = generar([d["entrada"] for d in demo], categorias=cats_demo)

for d, a, b, cat in zip(demo, gen_demo_A, gen_demo_B, cats_demo):
    print("=" * 78)
    print(f"[{d['id']}] {d['entrada']}")
    print(f"    categoría real: {d['categoria']}  |  predicha: {cat}")
    print("-" * 78)
    print("A · sin enrutamiento:")
    print(a[:450])
    print("-" * 78)
    print("B · con estrategia de categoría:")
    print(b[:450])
    print("-" * 78)
    print(f"Correcta -> A: {respuesta_correcta(a, d['valor'])} | "
          f"B: {respuesta_correcta(b, d['valor'])}")
print("=" * 78)

In [ ]:
RESULTADOS_PIPELINE = {
    "notebook": "S04_Lab_Fine_tuning_Qwen_BERT",
    "arquitectura": "pipeline modular (encoder + decoder)",
    "modelo": f"{DIR_BERT} -> {MODELO_QWEN}+LoRA",
    "metodo": "composición de modelos ya entrenados (sin entrenamiento nuevo)",
    "n_val": len(val),
    "umbral_confianza": UMBRAL_CONFIANZA,
    "accuracy_clasificador": accuracy_clf,
    "config_A_solo_qwen": metricas_A,
    "config_B_pipeline": metricas_B,
    "config_C_oraculo": metricas_C,
    "propagacion_error": {
        "n_clf_correcta": len(grupo_ok),
        "exactitud_clf_correcta": sum(grupo_ok) / max(len(grupo_ok), 1),
        "n_clf_incorrecta": len(grupo_mal),
        "exactitud_clf_incorrecta": sum(grupo_mal) / max(len(grupo_mal), 1),
    },
    "latencia_ms": {
        "clasificacion": t_clf * 1000,
        "generacion": t_gen * 1000,
        "total": (t_clf + t_gen) * 1000,
    },
    "beneficio_enrutamiento_perfecto": metricas_C["exactitud"] - metricas_A["exactitud"],
    "costo_errores_clasificador": metricas_C["exactitud"] - metricas_B["exactitud"],
    "aporte_neto": metricas_B["exactitud"] - metricas_A["exactitud"],
}

guardar_resultados("pipeline_qwen_bert", RESULTADOS_PIPELINE)

In [ ]:
if USAR_WANDB:
    import wandb

    if wandb.run is None:
        wandb.init(project=PROYECTO, name=NOMBRE_RUN, tags=TAGS, reinit=True)

    tabla = wandb.Table(columns=["id", "cat_real", "cat_predicha", "confianza",
                                 "clf_ok", "entrada", "gen_A", "gen_B",
                                 "ok_A", "ok_B"])
    for r, (cat, conf), a, b, oka, okb in zip(
            val, clasificaciones, gen_A, gen_B,
            metricas_A["_correctas"], metricas_B["_correctas"]):
        tabla.add_data(r["id"], r["categoria"], cat, conf,
                       cat == r["categoria"], r["entrada"], a, b, oka, okb)
    wandb.log({"evaluacion/pipeline": tabla})

    wandb.summary.update({
        "clasificador/accuracy": accuracy_clf,
        "A_solo_qwen/exactitud": metricas_A["exactitud"],
        "B_pipeline/exactitud": metricas_B["exactitud"],
        "C_oraculo/exactitud": metricas_C["exactitud"],
        "delta/aporte_neto": metricas_B["exactitud"] - metricas_A["exactitud"],
        "delta/costo_errores_clf": metricas_C["exactitud"] - metricas_B["exactitud"],
        "latencia/clasificacion_ms": t_clf * 1000,
        "latencia/generacion_ms": t_gen * 1000,
    })
    wandb.finish()
    print("Registro en W&B completado.")
else:
    print("W&B desactivado; las métricas quedaron en resultados/pipeline_qwen_bert.json")

## 13 · Discusión

**1. ¿El pipeline aporta algo? (B − A)**
Si la diferencia es positiva y apreciable, el enrutamiento funciona. Si es
cercana a cero o negativa, hay que decirlo sin adornos: el sistema modular
añadió complejidad sin beneficio medible en este corpus. Es un resultado
legítimo, y probablemente el más común con 33 ejemplos de evaluación.

**2. ¿El límite es el clasificador o la idea? (C vs A)**
- Si **C > A** de forma clara pero **B ≈ A**: la idea es buena y el
  clasificador es el cuello de botella. Invertir en más datos de clasificación
  tiene retorno.
- Si **C ≈ A**: condicionar por categoría no cambia lo que Qwen genera. Ningún
  clasificador, por perfecto que fuera, mejoraría el sistema. La conclusión es
  que el modelo fine-tuned ya infiere la estrategia del enunciado, y explicitarla
  es redundante.

**3. ¿Por qué el efecto podría ser pequeño?**
Una hipótesis razonable: Qwen ya vio 132 ejemplos con este formato y ya aprendió
implícitamente qué hacer con cada tipo de problema. Las estrategias explícitas
le dicen algo que ya sabía. El enrutamiento tendría más valor si las estrategias
aportaran conocimiento realmente ausente del entrenamiento —fórmulas poco
comunes, convenciones de notación, casos límite—.

**4. ¿Y la complejidad operativa?**
Dos modelos son dos artefactos que versionar, desplegar y monitorizar. La
alineación de etiquetas entre ambos (el `assert` de la sección 6) es un fallo
silencioso esperando ocurrir. Al comparar con FLAN-T5 en el notebook 4, este
costo cuenta tanto como las métricas.

## 14 · Conclusiones

1. **Modularidad tiene un valor real que las métricas no capturan:**
   diagnosticabilidad. Poder decir "el 12% de los fallos vienen del enrutador y
   el 88% del generador" es imposible en un modelo único, y es exactamente lo
   que se necesita para decidir dónde invertir el siguiente esfuerzo.
2. **La cascada propaga errores.** Está medido en este notebook, no supuesto.
   El umbral de confianza es la mitigación estándar y está implementado.
3. **El costo computacional del enrutamiento es despreciable.** El encoder es
   ~100 veces más barato que la generación.
4. **Cuál gana depende de sus números.** Comparen B contra el modelo único de
   FLAN-T5 (notebook 4), que resuelve clasificación y generación en una sola
   pasada, y contra A. La comparación completa está en el notebook 5.